# Interactive Neural Network Learning Platform
Name: Mariem Abidi

Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple, Union

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, mean_squared_error, r2_score
)

import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)

Activation functions

In [ ]:
class Activation:
    @staticmethod
    def sigmoid(z):
        z = np.clip(z, -500, 500)
        return 1.0 / (1.0 + np.exp(-z))

    @staticmethod
    def sigmoid_derivative(a):
        return a * (1 - a)

    @staticmethod
    def relu(z):
        return np.maximum(0, z)

    @staticmethod
    def relu_derivative(z):
        return (z > 0).astype(float)

    @staticmethod
    def tanh(z):
        return np.tanh(z)

    @staticmethod
    def tanh_derivative(a):
        return 1 - np.square(a)

    @staticmethod
    def softmax(z):
        z_shifted = z - np.max(z, axis=1, keepdims=True)
        exp_z = np.exp(z_shifted)
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)

    @staticmethod
    def linear(z):
        return z

    @staticmethod
    def linear_derivative(z):
        return np.ones_like(z)

    @staticmethod
    def get(name):
        name = name.lower()
        if name == "sigmoid":
            return Activation.sigmoid
        elif name == "relu":
            return Activation.relu
        elif name == "tanh":
            return Activation.tanh
        elif name == "softmax":
            return Activation.softmax
        elif name == "linear":
            return Activation.linear
        else:
            raise ValueError(f"Unsupported activation: {name}")

Loss functions

In [ ]:
class Loss:
    EPS = 1e-12

    @staticmethod
    def mse(y_true, y_pred):
        return np.mean((y_true - y_pred) ** 2)

    @staticmethod
    def mse_derivative(y_true, y_pred):
        return 2 * (y_pred - y_true) / y_true.shape[0]

    @staticmethod
    def binary_cross_entropy(y_true, y_pred):
        y_pred = np.clip(y_pred, Loss.EPS, 1 - Loss.EPS)
        return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

    @staticmethod
    def binary_cross_entropy_derivative(y_true, y_pred):
        y_pred = np.clip(y_pred, Loss.EPS, 1 - Loss.EPS)
        return (y_pred - y_true) / ((y_pred * (1 - y_pred)) * y_true.shape[0])

    @staticmethod
    def categorical_cross_entropy(y_true, y_pred):
        y_pred = np.clip(y_pred, Loss.EPS, 1 - Loss.EPS)
        return -np.mean(np.sum(y_true * np.log(y_pred), axis=1))

    @staticmethod
    def categorical_cross_entropy_derivative(y_true, y_pred):
        y_pred = np.clip(y_pred, Loss.EPS, 1 - Loss.EPS)
        return -(y_true / y_pred) / y_true.shape[0]

    @staticmethod
    def get(name):
        name = name.lower()
        if name == "mse":
            return Loss.mse
        elif name in ["binary_cross_entropy", "bce"]:
            return Loss.binary_cross_entropy
        elif name in ["categorical_cross_entropy", "cce"]:
            return Loss.categorical_cross_entropy
        else:
            raise ValueError(f"Unsupported loss: {name}")

Metrics

In [ ]:
class Metrics:
    @staticmethod
    def classification_metrics(y_true, y_pred, average="binary"):
        return {
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, average=average, zero_division=0),
            "recall": recall_score(y_true, y_pred, average=average, zero_division=0),
            "f1": f1_score(y_true, y_pred, average=average, zero_division=0),
            "confusion_matrix": confusion_matrix(y_true, y_pred)
        }

    @staticmethod
    def regression_metrics(y_true, y_pred):
        mse = mean_squared_error(y_true, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_true, y_pred)
        return {
            "mse": mse,
            "rmse": rmse,
            "r2": r2
        }

Data utilities

In [ ]:
class DataUtils:
    @staticmethod
    def load_csv(path: str) -> pd.DataFrame:
        return pd.read_csv(path)

    @staticmethod
    def split_features_target(df: pd.DataFrame, target_col: str):
        X = df.drop(columns=[target_col]).values
        y = df[target_col].values
        return X, y

    @staticmethod
    def standardize_train_test(X_train, X_test):
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        return X_train_scaled, X_test_scaled, scaler

    @staticmethod
    def prepare_binary_target(y):
        if y.ndim > 1:
            y = y.ravel()
        le = LabelEncoder()
        y_enc = le.fit_transform(y).astype(float).reshape(-1, 1)
        return y_enc, le

    @staticmethod
    def prepare_multiclass_target(y):
        if y.ndim > 1:
            y = y.ravel()
        le = LabelEncoder()
        y_int = le.fit_transform(y).reshape(-1, 1)
        encoder = OneHotEncoder(sparse_output=False)
        y_onehot = encoder.fit_transform(y_int)
        return y_onehot, le, encoder

    @staticmethod
    def prepare_regression_target(y):
        y = np.asarray(y).astype(float).reshape(-1, 1)
        return y

    @staticmethod
    def train_test_data(
        X, y, test_size=0.2, random_state=42, stratify=None
    ):
        return train_test_split(
            X, y, test_size=test_size, random_state=random_state, stratify=stratify
        )

Helper functions

In [ ]:
def one_hot_to_labels(y_onehot):
    return np.argmax(y_onehot, axis=1)

def probs_to_binary(y_prob, threshold=0.5):
    return (y_prob >= threshold).astype(int)

def generate_batches(X, y, batch_size=None, shuffle=True):
    n = X.shape[0]
    indices = np.arange(n)

    if shuffle:
        np.random.shuffle(indices)

    if batch_size is None or batch_size >= n:
        yield X[indices], y[indices]
        return

    for start_idx in range(0, n, batch_size):
        end_idx = start_idx + batch_size
        batch_idx = indices[start_idx:end_idx]
        yield X[batch_idx], y[batch_idx]

Historical perceptron

In [ ]:
class HistoricalPerceptron:
    """
    Classic perceptron with step activation.
    Binary classification only.
    Targets must be 0/1.
    """
    def __init__(self, learning_rate=0.01, epochs=100):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.weights = None
        self.bias = None
        self.history = {"errors": []}

    def step(self, z):
        return (z >= 0).astype(int)

    def fit(self, X, y):
        y = y.ravel().astype(int)
        n_samples, n_features = X.shape

        self.weights = np.zeros(n_features)
        self.bias = 0.0

        for _ in range(self.epochs):
            errors = 0
            for i in range(n_samples):
                linear_output = np.dot(X[i], self.weights) + self.bias
                y_pred = self.step(linear_output)
                update = self.learning_rate * (y[i] - y_pred)

                self.weights += update * X[i]
                self.bias += update

                errors += int(update != 0.0)

            self.history["errors"].append(errors)

        return self

    def predict(self, X):
        linear_output = np.dot(X, self.weights) + self.bias
        return self.step(linear_output).reshape(-1, 1)

    def predict_proba(self, X):
        preds = self.predict(X)
        return preds.astype(float)

Activation-based perceptron

In [ ]:
class ActivationPerceptron:
    """
    Single-layer perceptron with differentiable activation.
    Supports binary classification and regression.
    """
    def __init__(
        self,
        task_type="binary",
        activation="sigmoid",
        learning_rate=0.01,
        epochs=100,
        l2_lambda=0.0
    ):
        self.task_type = task_type
        self.activation_name = activation
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.l2_lambda = l2_lambda

        self.weights = None
        self.bias = None
        self.history = {
            "loss": [],
            "train_metric": []
        }

    def _initialize(self, n_features):
        self.weights = np.random.randn(n_features, 1) * 0.01
        self.bias = np.zeros((1, 1))

    def _forward(self, X):
        z = X @ self.weights + self.bias
        if self.task_type == "binary":
            a = Activation.sigmoid(z)
        elif self.task_type == "regression":
            a = z
        else:
            raise ValueError("ActivationPerceptron supports only binary or regression.")
        return z, a

    def _compute_loss(self, y, y_pred):
        if self.task_type == "binary":
            base_loss = Loss.binary_cross_entropy(y, y_pred)
        else:
            base_loss = Loss.mse(y, y_pred)

        reg_loss = (self.l2_lambda / (2 * y.shape[0])) * np.sum(np.square(self.weights))
        return base_loss + reg_loss

    def fit(self, X, y):
        y = y.reshape(-1, 1)
        n_samples, n_features = X.shape
        self._initialize(n_features)

        for _ in range(self.epochs):
            z, y_pred = self._forward(X)

            if self.task_type == "binary":
                dz = (y_pred - y) / n_samples
            else:
                dz = 2 * (y_pred - y) / n_samples

            dw = X.T @ dz + (self.l2_lambda / n_samples) * self.weights
            db = np.sum(dz, axis=0, keepdims=True)

            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db

            loss = self._compute_loss(y, y_pred)
            self.history["loss"].append(loss)

            if self.task_type == "binary":
                preds = (y_pred >= 0.5).astype(int)
                acc = accuracy_score(y, preds)
                self.history["train_metric"].append(acc)
            else:
                mse_val = mean_squared_error(y, y_pred)
                self.history["train_metric"].append(mse_val)

        return self

    def predict_proba(self, X):
        _, y_pred = self._forward(X)
        return y_pred

    def predict(self, X):
        y_pred = self.predict_proba(X)
        if self.task_type == "binary":
            return (y_pred >= 0.5).astype(int)
        return y_pred

Full MLP

In [ ]:
class MLP:
    """
    Flexible MLP supporting:
    - binary classification
    - multi-class classification
    - regression
    - multiple hidden layers
    - L2 regularization
    - mini-batch gradient descent
    - early stopping
    """
    def __init__(
        self,
        input_dim: int,
        hidden_layers: List[int],
        output_dim: int,
        task_type: str,
        hidden_activation: str = "relu",
        learning_rate: float = 0.01,
        epochs: int = 100,
        batch_size: Optional[int] = None,
        l2_lambda: float = 0.0,
        early_stopping: bool = False,
        patience: int = 20,
        min_delta: float = 1e-4,
        verbose: bool = True
    ):
        self.input_dim = input_dim
        self.hidden_layers = hidden_layers
        self.output_dim = output_dim
        self.task_type = task_type
        self.hidden_activation = hidden_activation.lower()
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.l2_lambda = l2_lambda
        self.early_stopping = early_stopping
        self.patience = patience
        self.min_delta = min_delta
        self.verbose = verbose

        self.layer_dims = [input_dim] + hidden_layers + [output_dim]
        self.weights = {}
        self.biases = {}
        self.history = {
            "train_loss": [],
            "val_loss": [],
            "train_metric": [],
            "val_metric": []
        }

        self._initialize_parameters()

    def _initialize_parameters(self):
        for l in range(1, len(self.layer_dims)):
            fan_in = self.layer_dims[l - 1]
            fan_out = self.layer_dims[l]

            if self.hidden_activation == "relu":
                limit = np.sqrt(2 / fan_in)
            else:
                limit = np.sqrt(1 / fan_in)

            self.weights[l] = np.random.randn(fan_in, fan_out) * limit
            self.biases[l] = np.zeros((1, fan_out))

    def _hidden_activation_forward(self, z):
        if self.hidden_activation == "sigmoid":
            return Activation.sigmoid(z)
        elif self.hidden_activation == "relu":
            return Activation.relu(z)
        elif self.hidden_activation == "tanh":
            return Activation.tanh(z)
        else:
            raise ValueError(f"Unsupported hidden activation: {self.hidden_activation}")

    def _hidden_activation_backward(self, z, a):
        if self.hidden_activation == "sigmoid":
            return Activation.sigmoid_derivative(a)
        elif self.hidden_activation == "relu":
            return Activation.relu_derivative(z)
        elif self.hidden_activation == "tanh":
            return Activation.tanh_derivative(a)
        else:
            raise ValueError(f"Unsupported hidden activation: {self.hidden_activation}")

    def _output_activation(self, z):
        if self.task_type == "binary":
            return Activation.sigmoid(z)
        elif self.task_type == "multiclass":
            return Activation.softmax(z)
        elif self.task_type == "regression":
            return Activation.linear(z)
        else:
            raise ValueError(f"Unsupported task type: {self.task_type}")

    def _compute_loss(self, y_true, y_pred):
        if self.task_type == "binary":
            base_loss = Loss.binary_cross_entropy(y_true, y_pred)
        elif self.task_type == "multiclass":
            base_loss = Loss.categorical_cross_entropy(y_true, y_pred)
        elif self.task_type == "regression":
            base_loss = Loss.mse(y_true, y_pred)
        else:
            raise ValueError(f"Unsupported task type: {self.task_type}")

        reg_loss = 0.0
        for l in self.weights:
            reg_loss += np.sum(np.square(self.weights[l]))
        reg_loss *= self.l2_lambda / (2 * y_true.shape[0])

        return base_loss + reg_loss

    def forward(self, X):
        cache = {"A0": X}
        A = X

        for l in range(1, len(self.layer_dims) - 1):
            Z = A @ self.weights[l] + self.biases[l]
            A = self._hidden_activation_forward(Z)
            cache[f"Z{l}"] = Z
            cache[f"A{l}"] = A

        L = len(self.layer_dims) - 1
        ZL = A @ self.weights[L] + self.biases[L]
        AL = self._output_activation(ZL)

        cache[f"Z{L}"] = ZL
        cache[f"A{L}"] = AL

        return AL, cache

    def backward(self, y_true, cache):
        grads = {}
        m = y_true.shape[0]
        L = len(self.layer_dims) - 1
        AL = cache[f"A{L}"]

        if self.task_type in ["binary", "multiclass"]:
            dZ = AL - y_true
        elif self.task_type == "regression":
            dZ = 2 * (AL - y_true)
        else:
            raise ValueError(f"Unsupported task type: {self.task_type}")

        grads[f"dW{L}"] = (cache[f"A{L-1}"].T @ dZ) / m + (self.l2_lambda / m) * self.weights[L]
        grads[f"db{L}"] = np.sum(dZ, axis=0, keepdims=True) / m

        dA_prev = dZ @ self.weights[L].T

        for l in reversed(range(1, L)):
            Z = cache[f"Z{l}"]
            A = cache[f"A{l}"]
            A_prev = cache[f"A{l-1}"]

            d_activation = self._hidden_activation_backward(Z, A)
            dZ = dA_prev * d_activation

            grads[f"dW{l}"] = (A_prev.T @ dZ) / m + (self.l2_lambda / m) * self.weights[l]
            grads[f"db{l}"] = np.sum(dZ, axis=0, keepdims=True) / m

            if l > 1:
                dA_prev = dZ @ self.weights[l].T

        return grads

    def _update_parameters(self, grads):
        L = len(self.layer_dims) - 1
        for l in range(1, L + 1):
            self.weights[l] -= self.learning_rate * grads[f"dW{l}"]
            self.biases[l] -= self.learning_rate * grads[f"db{l}"]

    def _compute_metric(self, y_true, y_pred):
        if self.task_type == "binary":
            y_labels = (y_pred >= 0.5).astype(int)
            return accuracy_score(y_true, y_labels)
        elif self.task_type == "multiclass":
            y_true_labels = np.argmax(y_true, axis=1)
            y_pred_labels = np.argmax(y_pred, axis=1)
            return accuracy_score(y_true_labels, y_pred_labels)
        elif self.task_type == "regression":
            return mean_squared_error(y_true, y_pred)
        else:
            raise ValueError(f"Unsupported task type: {self.task_type}")

    def fit(self, X_train, y_train, X_val=None, y_val=None):
        best_val_loss = np.inf
        patience_counter = 0
        best_weights = None
        best_biases = None

        for epoch in range(self.epochs):
            for X_batch, y_batch in generate_batches(
                X_train, y_train, batch_size=self.batch_size, shuffle=True
            ):
                y_pred_batch, cache = self.forward(X_batch)
                grads = self.backward(y_batch, cache)
                self._update_parameters(grads)

            train_pred, _ = self.forward(X_train)
            train_loss = self._compute_loss(y_train, train_pred)
            train_metric = self._compute_metric(y_train, train_pred)

            self.history["train_loss"].append(train_loss)
            self.history["train_metric"].append(train_metric)

            if X_val is not None and y_val is not None:
                val_pred, _ = self.forward(X_val)
                val_loss = self._compute_loss(y_val, val_pred)
                val_metric = self._compute_metric(y_val, val_pred)

                self.history["val_loss"].append(val_loss)
                self.history["val_metric"].append(val_metric)

                if self.early_stopping:
                    if val_loss < best_val_loss - self.min_delta:
                        best_val_loss = val_loss
                        patience_counter = 0
                        best_weights = {k: v.copy() for k, v in self.weights.items()}
                        best_biases = {k: v.copy() for k, v in self.biases.items()}
                    else:
                        patience_counter += 1

                    if patience_counter >= self.patience:
                        if self.verbose:
                            print(f"Early stopping at epoch {epoch + 1}")
                        if best_weights is not None:
                            self.weights = best_weights
                            self.biases = best_biases
                        break

            if self.verbose and ((epoch + 1) % max(1, self.epochs // 10) == 0 or epoch == 0):
                msg = f"Epoch {epoch+1}/{self.epochs} | Train Loss: {train_loss:.4f}"
                if self.task_type in ["binary", "multiclass"]:
                    msg += f" | Train Acc: {train_metric:.4f}"
                else:
                    msg += f" | Train MSE: {train_metric:.4f}"

                if X_val is not None and y_val is not None:
                    msg += f" | Val Loss: {val_loss:.4f}"
                    if self.task_type in ["binary", "multiclass"]:
                        msg += f" | Val Acc: {val_metric:.4f}"
                    else:
                        msg += f" | Val MSE: {val_metric:.4f}"
                print(msg)

        return self

    def predict_proba(self, X):
        y_pred, _ = self.forward(X)
        return y_pred

    def predict(self, X):
        y_pred = self.predict_proba(X)

        if self.task_type == "binary":
            return (y_pred >= 0.5).astype(int)
        elif self.task_type == "multiclass":
            return np.argmax(y_pred, axis=1)
        elif self.task_type == "regression":
            return y_pred
        else:
            raise ValueError(f"Unsupported task type: {self.task_type}")

Evaluation utilities

In [ ]:
def evaluate_model(model, X_test, y_test, task_type):
    y_pred_raw = model.predict_proba(X_test)

    if task_type == "binary":
        y_pred = (y_pred_raw >= 0.5).astype(int).ravel()
        y_true = y_test.ravel().astype(int)
        results = Metrics.classification_metrics(y_true, y_pred, average="binary")

    elif task_type == "multiclass":
        y_pred = np.argmax(y_pred_raw, axis=1)
        y_true = np.argmax(y_test, axis=1)
        results = Metrics.classification_metrics(y_true, y_pred, average="macro")

    elif task_type == "regression":
        y_pred = y_pred_raw.ravel()
        y_true = y_test.ravel()
        results = Metrics.regression_metrics(y_true, y_pred)

    else:
        raise ValueError(f"Unsupported task type: {task_type}")

    return results

Visualization utilities

In [ ]:
class Visualizer:
    @staticmethod
    def plot_training_history(history, task_type="binary"):
        plt.figure(figsize=(14, 5))

        plt.subplot(1, 2, 1)
        plt.plot(history["train_loss"], label="Train Loss")
        if len(history.get("val_loss", [])) > 0:
            plt.plot(history["val_loss"], label="Val Loss")
        plt.title("Loss vs Epochs")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.legend()
        plt.grid(True)

        plt.subplot(1, 2, 2)
        plt.plot(history["train_metric"], label="Train Metric")
        if len(history.get("val_metric", [])) > 0:
            plt.plot(history["val_metric"], label="Val Metric")

        if task_type in ["binary", "multiclass"]:
            plt.title("Accuracy vs Epochs")
            plt.ylabel("Accuracy")
        else:
            plt.title("MSE vs Epochs")
            plt.ylabel("MSE")

        plt.xlabel("Epoch")
        plt.legend()
        plt.grid(True)

        plt.tight_layout()
        plt.show()

    @staticmethod
    def plot_confusion_matrix(cm, title="Confusion Matrix"):
        plt.figure(figsize=(6, 5))
        plt.imshow(cm, interpolation="nearest")
        plt.title(title)
        plt.colorbar()
        plt.xlabel("Predicted")
        plt.ylabel("True")

        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                plt.text(j, i, str(cm[i, j]), ha="center", va="center")

        plt.tight_layout()
        plt.show()

    @staticmethod
    def plot_regression_predictions(y_true, y_pred, title="Predictions vs True"):
        plt.figure(figsize=(6, 6))
        plt.scatter(y_true, y_pred, alpha=0.7)
        min_val = min(np.min(y_true), np.min(y_pred))
        max_val = max(np.max(y_true), np.max(y_pred))
        plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")
        plt.xlabel("True Values")
        plt.ylabel("Predicted Values")
        plt.title(title)
        plt.grid(True)
        plt.show()

    @staticmethod
    def plot_decision_boundary(model, X, y, task_type="binary", title="Decision Boundary"):
        if X.shape[1] != 2:
            raise ValueError("Decision boundary plot requires exactly 2 input features.")

        x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
        y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
        xx, yy = np.meshgrid(
            np.linspace(x_min, x_max, 300),
            np.linspace(y_min, y_max, 300)
        )

        grid = np.c_[xx.ravel(), yy.ravel()]

        if task_type == "binary":
            Z = model.predict(grid).reshape(xx.shape)
        elif task_type == "multiclass":
            Z = model.predict(grid).reshape(xx.shape)
        else:
            raise ValueError("Decision boundary is only for classification.")

        plt.figure(figsize=(7, 6))
        plt.contourf(xx, yy, Z, alpha=0.35)

        if y.ndim > 1 and y.shape[1] > 1:
            y_plot = np.argmax(y, axis=1)
        else:
            y_plot = y.ravel()

        plt.scatter(X[:, 0], X[:, 1], c=y_plot, edgecolor="k", alpha=0.8)
        plt.title(title)
        plt.xlabel("Feature 1")
        plt.ylabel("Feature 2")
        plt.grid(True)
        plt.show()

Synthetic datasets for testing

In [ ]:
from sklearn.datasets import make_classification, make_moons, make_regression

def get_binary_dataset(dataset_type="moons", n_samples=500, noise=0.2, random_state=42):
    if dataset_type == "moons":
        X, y = make_moons(n_samples=n_samples, noise=noise, random_state=random_state)
    elif dataset_type == "classification":
        X, y = make_classification(
            n_samples=n_samples,
            n_features=2,
            n_redundant=0,
            n_informative=2,
            n_clusters_per_class=1,
            random_state=random_state
        )
    else:
        raise ValueError("Unsupported dataset_type.")
    return X, y.reshape(-1, 1)

def get_multiclass_dataset(n_samples=600, random_state=42):
    X, y = make_classification(
        n_samples=n_samples,
        n_features=2,
        n_redundant=0,
        n_informative=2,
        n_classes=3,
        n_clusters_per_class=1,
        random_state=random_state
    )
    return X, y

def get_regression_dataset(n_samples=500, noise=10.0, random_state=42):
    X, y = make_regression(
        n_samples=n_samples,
        n_features=4,
        noise=noise,
        random_state=random_state
    )
    return X, y.reshape(-1, 1)

Binary classification test with MLP

In [ ]:
# Binary classification example
X, y = get_binary_dataset(dataset_type="moons", n_samples=500, noise=0.25)

X_train, X_test, y_train, y_test = DataUtils.train_test_data(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train, X_test, scaler = DataUtils.standardize_train_test(X_train, X_test)

model = MLP(
    input_dim=X_train.shape[1],
    hidden_layers=[16, 16],
    output_dim=1,
    task_type="binary",
    hidden_activation="relu",
    learning_rate=0.05,
    epochs=300,
    batch_size=32,
    l2_lambda=0.001,
    early_stopping=True,
    patience=30,
    verbose=True
)

model.fit(X_train, y_train, X_val=X_test, y_val=y_test)

results = evaluate_model(model, X_test, y_test, task_type="binary")
print(results)

Visualizer.plot_training_history(model.history, task_type="binary")
Visualizer.plot_confusion_matrix(results["confusion_matrix"], title="Binary Confusion Matrix")
Visualizer.plot_decision_boundary(model, X_train, y_train, task_type="binary", title="Binary Decision Boundary")

Historical perceptron test

In [ ]:
# Historical perceptron on a simpler binary dataset
X, y = get_binary_dataset(dataset_type="classification", n_samples=400)

X_train, X_test, y_train, y_test = DataUtils.train_test_data(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train, X_test, scaler = DataUtils.standardize_train_test(X_train, X_test)

perceptron = HistoricalPerceptron(learning_rate=0.01, epochs=50)
perceptron.fit(X_train, y_train)

y_pred = perceptron.predict(X_test).ravel()
results = Metrics.classification_metrics(y_test.ravel(), y_pred, average="binary")
print(results)

plt.figure(figsize=(7, 4))
plt.plot(perceptron.history["errors"])
plt.title("Historical Perceptron Errors per Epoch")
plt.xlabel("Epoch")
plt.ylabel("Errors")
plt.grid(True)
plt.show()

Visualizer.plot_confusion_matrix(results["confusion_matrix"], title="Historical Perceptron Confusion Matrix")
Visualizer.plot_decision_boundary(perceptron, X_train, y_train, task_type="binary", title="Historical Perceptron Decision Boundary")

Multi-class classification test

In [ ]:
# Multi-class classification example
X, y_raw = get_multiclass_dataset(n_samples=600)

X_train, X_test, y_train_raw, y_test_raw = train_test_split(
    X, y_raw, test_size=0.2, random_state=42, stratify=y_raw
)

X_train, X_test, scaler = DataUtils.standardize_train_test(X_train, X_test)

y_train, le, ohe = DataUtils.prepare_multiclass_target(y_train_raw)
y_test = ohe.transform(y_test_raw.reshape(-1, 1))

model_mc = MLP(
    input_dim=X_train.shape[1],
    hidden_layers=[32, 16],
    output_dim=y_train.shape[1],
    task_type="multiclass",
    hidden_activation="tanh",
    learning_rate=0.05,
    epochs=300,
    batch_size=32,
    l2_lambda=0.001,
    early_stopping=True,
    patience=30,
    verbose=True
)

model_mc.fit(X_train, y_train, X_val=X_test, y_val=y_test)

results_mc = evaluate_model(model_mc, X_test, y_test, task_type="multiclass")
print(results_mc)

Visualizer.plot_training_history(model_mc.history, task_type="multiclass")
Visualizer.plot_confusion_matrix(results_mc["confusion_matrix"], title="Multi-class Confusion Matrix")
Visualizer.plot_decision_boundary(model_mc, X_train, y_train, task_type="multiclass", title="Multi-class Decision Boundary")

Regression test

In [ ]:
# Regression example
X, y = get_regression_dataset(n_samples=500, noise=20.0)

X_train, X_test, y_train, y_test = DataUtils.train_test_data(
    X, y, test_size=0.2, random_state=42
)

X_train, X_test, scaler = DataUtils.standardize_train_test(X_train, X_test)

model_reg = MLP(
    input_dim=X_train.shape[1],
    hidden_layers=[32, 16],
    output_dim=1,
    task_type="regression",
    hidden_activation="relu",
    learning_rate=0.01,
    epochs=400,
    batch_size=32,
    l2_lambda=0.001,
    early_stopping=True,
    patience=40,
    verbose=True
)

model_reg.fit(X_train, y_train, X_val=X_test, y_val=y_test)

results_reg = evaluate_model(model_reg, X_test, y_test, task_type="regression")
print(results_reg)

Visualizer.plot_training_history(model_reg.history, task_type="regression")

y_pred_reg = model_reg.predict(X_test).ravel()
Visualizer.plot_regression_predictions(y_test.ravel(), y_pred_reg, title="Regression Predictions")

CSV pipeline helper

In [ ]:
def prepare_dataset_from_dataframe(
    df: pd.DataFrame,
    target_col: str,
    task_type: str,
    test_size: float = 0.2,
    random_state: int = 42,
    standardize: bool = True
):
    X, y = DataUtils.split_features_target(df, target_col)

    label_encoder = None
    onehot_encoder = None
    scaler = None

    if task_type == "binary":
        y_prepared, label_encoder = DataUtils.prepare_binary_target(y)
        stratify = y_prepared.ravel()

    elif task_type == "multiclass":
        y_prepared, label_encoder, onehot_encoder = DataUtils.prepare_multiclass_target(y)
        stratify = np.argmax(y_prepared, axis=1)

    elif task_type == "regression":
        y_prepared = DataUtils.prepare_regression_target(y)
        stratify = None

    else:
        raise ValueError("task_type must be one of: binary, multiclass, regression")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y_prepared, test_size=test_size, random_state=random_state, stratify=stratify
    )

    if standardize:
        X_train, X_test, scaler = DataUtils.standardize_train_test(X_train, X_test)

    metadata = {
        "label_encoder": label_encoder,
        "onehot_encoder": onehot_encoder,
        "scaler": scaler,
        "feature_names": df.drop(columns=[target_col]).columns.tolist(),
        "target_col": target_col,
        "task_type": task_type
    }

    return X_train, X_test, y_train, y_test, metadata

Train from CSV dataframe

In [ ]:
def build_and_train_from_dataframe(
    df: pd.DataFrame,
    target_col: str,
    task_type: str,
    hidden_layers: List[int],
    hidden_activation: str = "relu",
    learning_rate: float = 0.01,
    epochs: int = 200,
    batch_size: Optional[int] = 32,
    l2_lambda: float = 0.0,
    early_stopping: bool = True,
    patience: int = 20,
    test_size: float = 0.2,
    standardize: bool = True,
    verbose: bool = True
):
    X_train, X_test, y_train, y_test, metadata = prepare_dataset_from_dataframe(
        df=df,
        target_col=target_col,
        task_type=task_type,
        test_size=test_size,
        standardize=standardize
    )

    output_dim = 1 if task_type in ["binary", "regression"] else y_train.shape[1]

    model = MLP(
        input_dim=X_train.shape[1],
        hidden_layers=hidden_layers,
        output_dim=output_dim,
        task_type=task_type,
        hidden_activation=hidden_activation,
        learning_rate=learning_rate,
        epochs=epochs,
        batch_size=batch_size,
        l2_lambda=l2_lambda,
        early_stopping=early_stopping,
        patience=patience,
        verbose=verbose
    )

    model.fit(X_train, y_train, X_val=X_test, y_val=y_test)
    results = evaluate_model(model, X_test, y_test, task_type)

    return model, results, X_train, X_test, y_train, y_test, metadata

Example CSV usage

In [ ]:
# Example usage after uploading a CSV manually:
# df = pd.read_csv("your_file.csv")

# model, results, X_train, X_test, y_train, y_test, metadata = build_and_train_from_dataframe(
#     df=df,
#     target_col="target",
#     task_type="binary",   # "binary", "multiclass", or "regression"
#     hidden_layers=[16, 8],
#     hidden_activation="relu",
#     learning_rate=0.01,
#     epochs=200,
#     batch_size=32,
#     l2_lambda=0.001,
#     early_stopping=True,
#     patience=20,
#     standardize=True,
#     verbose=True
# )

# print(results)
# Visualizer.plot_training_history(model.history, task_type="binary")

Experiment 1: Perceptron vs MLP

In [ ]:
# Experiment 1: Perceptron vs MLP
X, y = get_binary_dataset(dataset_type="classification", n_samples=500)

X_train, X_test, y_train, y_test = DataUtils.train_test_data(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_test, scaler = DataUtils.standardize_train_test(X_train, X_test)

# Historical perceptron
hist_p = HistoricalPerceptron(learning_rate=0.01, epochs=50)
hist_p.fit(X_train, y_train)
hist_results = Metrics.classification_metrics(y_test.ravel(), hist_p.predict(X_test).ravel(), average="binary")

# MLP
mlp = MLP(
    input_dim=2,
    hidden_layers=[16, 16],
    output_dim=1,
    task_type="binary",
    hidden_activation="relu",
    learning_rate=0.05,
    epochs=200,
    batch_size=32,
    l2_lambda=0.001,
    verbose=False
)
mlp.fit(X_train, y_train, X_val=X_test, y_val=y_test)
mlp_results = evaluate_model(mlp, X_test, y_test, task_type="binary")

print("Historical Perceptron:", hist_results)
print("MLP:", mlp_results)

Visualizer.plot_decision_boundary(hist_p, X_train, y_train, task_type="binary", title="Historical Perceptron Boundary")
Visualizer.plot_decision_boundary(mlp, X_train, y_train, task_type="binary", title="MLP Boundary")

Experiment 2: Effect of number of layers

In [ ]:
# Experiment 2: Effect of depth
X, y = get_binary_dataset(dataset_type="moons", n_samples=500, noise=0.25)

X_train, X_test, y_train, y_test = DataUtils.train_test_data(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_test, scaler = DataUtils.standardize_train_test(X_train, X_test)

architectures = {
    "1 hidden layer": [16],
    "2 hidden layers": [16, 16],
    "3 hidden layers": [16, 16, 16]
}

all_results = {}

for name, arch in architectures.items():
    model = MLP(
        input_dim=2,
        hidden_layers=arch,
        output_dim=1,
        task_type="binary",
        hidden_activation="relu",
        learning_rate=0.05,
        epochs=250,
        batch_size=32,
        l2_lambda=0.001,
        verbose=False
    )
    model.fit(X_train, y_train, X_val=X_test, y_val=y_test)
    res = evaluate_model(model, X_test, y_test, task_type="binary")
    all_results[name] = res
    print(name, res)

Experiment 3: Overfitting vs regularization

In [ ]:
# Experiment 3: Overfitting vs L2 regularization
X, y = get_binary_dataset(dataset_type="moons", n_samples=700, noise=0.30)

X_train, X_test, y_train, y_test = DataUtils.train_test_data(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_test, scaler = DataUtils.standardize_train_test(X_train, X_test)

model_no_reg = MLP(
    input_dim=2,
    hidden_layers=[64, 64, 64],
    output_dim=1,
    task_type="binary",
    hidden_activation="relu",
    learning_rate=0.05,
    epochs=300,
    batch_size=32,
    l2_lambda=0.0,
    verbose=False
)

model_reg = MLP(
    input_dim=2,
    hidden_layers=[64, 64, 64],
    output_dim=1,
    task_type="binary",
    hidden_activation="relu",
    learning_rate=0.05,
    epochs=300,
    batch_size=32,
    l2_lambda=0.01,
    verbose=False
)

model_no_reg.fit(X_train, y_train, X_val=X_test, y_val=y_test)
model_reg.fit(X_train, y_train, X_val=X_test, y_val=y_test)

res_no_reg = evaluate_model(model_no_reg, X_test, y_test, task_type="binary")
res_reg = evaluate_model(model_reg, X_test, y_test, task_type="binary")

print("No Regularization:", res_no_reg)
print("With L2:", res_reg)

plt.figure(figsize=(10, 5))
plt.plot(model_no_reg.history["train_loss"], label="Train Loss - No Reg")
plt.plot(model_no_reg.history["val_loss"], label="Val Loss - No Reg")
plt.plot(model_reg.history["train_loss"], label="Train Loss - L2")
plt.plot(model_reg.history["val_loss"], label="Val Loss - L2")
plt.title("Overfitting vs Regularization")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

the widget builder

In [ ]:
def create_training_widgets(df: pd.DataFrame):
    columns = df.columns.tolist()

    target_widget = widgets.Dropdown(
        options=columns,
        description="Target:",
        layout=widgets.Layout(width="420px")
    )

    suggested_task = suggest_task_type(df, columns[-1]) if len(columns) > 0 else "binary"

    task_widget = widgets.Dropdown(
        options=["binary", "multiclass", "regression"],
        value=suggested_task,
        description="Task:",
        layout=widgets.Layout(width="420px")
    )

    model_widget = widgets.Dropdown(
        options=["historical_perceptron", "activation_perceptron", "mlp"],
        value="mlp",
        description="Model:",
        layout=widgets.Layout(width="420px")
    )

    hidden_layers_widget = widgets.Text(
        value="16,16",
        description="Hidden:",
        layout=widgets.Layout(width="420px")
    )

    activation_widget = widgets.Dropdown(
        options=["relu", "sigmoid", "tanh"],
        value="relu",
        description="Activation:",
        layout=widgets.Layout(width="420px")
    )

    learning_rate_widget = widgets.FloatText(
        value=0.01,
        description="LR:",
        layout=widgets.Layout(width="420px")
    )

    epochs_widget = widgets.IntText(
        value=200,
        description="Epochs:",
        layout=widgets.Layout(width="420px")
    )

    batch_size_widget = widgets.IntText(
        value=32,
        description="Batch:",
        layout=widgets.Layout(width="420px")
    )

    l2_widget = widgets.FloatText(
        value=0.001,
        description="L2:",
        layout=widgets.Layout(width="420px")
    )

    patience_widget = widgets.IntText(
        value=20,
        description="Patience:",
        layout=widgets.Layout(width="420px")
    )

    test_size_widget = widgets.FloatSlider(
        value=0.2,
        min=0.1,
        max=0.4,
        step=0.05,
        description="Test Size:",
        layout=widgets.Layout(width="500px")
    )

    standardize_widget = widgets.Checkbox(
        value=True,
        description="Standardize features"
    )

    early_stopping_widget = widgets.Checkbox(
        value=True,
        description="Use early stopping"
    )

    verbose_widget = widgets.Checkbox(
        value=True,
        description="Verbose training"
    )

    sklearn_compare_widget = widgets.Checkbox(
        value=True,
        description="Compare with sklearn"
    )

    run_button = widgets.Button(
        description="Train Model",
        button_style="success",
        icon="play"
    )

    compare_button = widgets.Button(
        description="Run Architecture Comparison",
        button_style="info",
        icon="bar-chart"
    )

    widgets_dict = {
        "target": target_widget,
        "task": task_widget,
        "model": model_widget,
        "hidden_layers": hidden_layers_widget,
        "activation": activation_widget,
        "learning_rate": learning_rate_widget,
        "epochs": epochs_widget,
        "batch_size": batch_size_widget,
        "l2_lambda": l2_widget,
        "patience": patience_widget,
        "test_size": test_size_widget,
        "standardize": standardize_widget,
        "early_stopping": early_stopping_widget,
        "verbose": verbose_widget,
        "sklearn_compare": sklearn_compare_widget,
        "run_button": run_button,
        "compare_button": compare_button,
    }

    return widgets_dict

Display widgets cleanly

In [ ]:
def display_training_widgets(w):
    ui = widgets.VBox([
        widgets.HTML("<h3>Model Configuration</h3>"),
        w["target"],
        w["task"],
        w["model"],
        w["hidden_layers"],
        w["activation"],
        w["learning_rate"],
        w["epochs"],
        w["batch_size"],
        w["l2_lambda"],
        w["patience"],
        w["test_size"],
        w["standardize"],
        w["early_stopping"],
        w["verbose"],
        w["sklearn_compare"],
        widgets.HBox([w["run_button"], w["compare_button"]])
    ])
    display(ui)

Parse hidden-layer input

In [ ]:
def parse_hidden_layers(text_value: str):
    text_value = text_value.strip()
    if text_value == "":
        return []

    layers = []
    for x in text_value.split(","):
        x = x.strip()
        if x:
            layers.append(int(x))
    return layers

Training config extractor

In [ ]:
def extract_config_from_widgets(w):
    config = {
        "target_col": w["target"].value,
        "task_type": w["task"].value,
        "model_type": w["model"].value,
        "hidden_layers": parse_hidden_layers(w["hidden_layers"].value),
        "hidden_activation": w["activation"].value,
        "learning_rate": float(w["learning_rate"].value),
        "epochs": int(w["epochs"].value),
        "batch_size": int(w["batch_size"].value),
        "l2_lambda": float(w["l2_lambda"].value),
        "patience": int(w["patience"].value),
        "test_size": float(w["test_size"].value),
        "standardize": bool(w["standardize"].value),
        "early_stopping": bool(w["early_stopping"].value),
        "verbose": bool(w["verbose"].value),
        "sklearn_compare": bool(w["sklearn_compare"].value),
    }
    return config

Sklearn bonus imports

In [ ]:
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.linear_model import Perceptron as SklearnPerceptron, LogisticRegression

Sklearn comparison engine

In [ ]:
def run_sklearn_comparison(X_train, X_test, y_train, y_test, config):
    task_type = config["task_type"]
    model_type = config["model_type"]
    hidden_layers = tuple(config["hidden_layers"]) if len(config["hidden_layers"]) > 0 else ()
    activation = config["hidden_activation"]
    learning_rate = config["learning_rate"]
    epochs = config["epochs"]

    results = {}

    if task_type == "binary":
        y_train_flat = y_train.ravel().astype(int)
        y_test_flat = y_test.ravel().astype(int)

        if model_type == "historical_perceptron":
            sk_model = SklearnPerceptron(
                eta0=learning_rate,
                max_iter=epochs,
                random_state=42,
                tol=1e-4
            )
            sk_model.fit(X_train, y_train_flat)
            y_pred = sk_model.predict(X_test)
            results = Metrics.classification_metrics(y_test_flat, y_pred, average="binary")

        elif model_type == "activation_perceptron":
            sk_model = LogisticRegression(
                max_iter=epochs,
                random_state=42
            )
            sk_model.fit(X_train, y_train_flat)
            y_pred = sk_model.predict(X_test)
            results = Metrics.classification_metrics(y_test_flat, y_pred, average="binary")

        elif model_type == "mlp":
            act = "logistic" if activation == "sigmoid" else activation
            sk_model = MLPClassifier(
                hidden_layer_sizes=hidden_layers,
                activation=act,
                learning_rate_init=learning_rate,
                max_iter=epochs,
                alpha=config["l2_lambda"],
                batch_size=config["batch_size"],
                random_state=42
            )
            sk_model.fit(X_train, y_train_flat)
            y_pred = sk_model.predict(X_test)
            results = Metrics.classification_metrics(y_test_flat, y_pred, average="binary")

    elif task_type == "multiclass":
        y_train_flat = np.argmax(y_train, axis=1)
        y_test_flat = np.argmax(y_test, axis=1)

        act = "logistic" if activation == "sigmoid" else activation
        sk_model = MLPClassifier(
            hidden_layer_sizes=hidden_layers,
            activation=act,
            learning_rate_init=learning_rate,
            max_iter=epochs,
            alpha=config["l2_lambda"],
            batch_size=config["batch_size"],
            random_state=42
        )
        sk_model.fit(X_train, y_train_flat)
        y_pred = sk_model.predict(X_test)
        results = Metrics.classification_metrics(y_test_flat, y_pred, average="macro")

    elif task_type == "regression":
        y_train_flat = y_train.ravel()
        y_test_flat = y_test.ravel()

        if model_type in ["historical_perceptron"]:
            return {"error": "Historical perceptron does not support regression in sklearn comparison."}

        elif model_type == "activation_perceptron":
            sk_model = MLPRegressor(
                hidden_layer_sizes=(),
                activation="identity",
                learning_rate_init=learning_rate,
                max_iter=epochs,
                alpha=config["l2_lambda"],
                batch_size=config["batch_size"],
                random_state=42
            )
            sk_model.fit(X_train, y_train_flat)
            y_pred = sk_model.predict(X_test)
            results = Metrics.regression_metrics(y_test_flat, y_pred)

        elif model_type == "mlp":
            act = "logistic" if activation == "sigmoid" else activation
            sk_model = MLPRegressor(
                hidden_layer_sizes=hidden_layers,
                activation=act,
                learning_rate_init=learning_rate,
                max_iter=epochs,
                alpha=config["l2_lambda"],
                batch_size=config["batch_size"],
                random_state=42
            )
            sk_model.fit(X_train, y_train_flat)
            y_pred = sk_model.predict(X_test)
            results = Metrics.regression_metrics(y_test_flat, y_pred)

    return results

Side-by-side comparison display

In [ ]:
def display_comparison_results(custom_results, sklearn_results, task_type):
    print("===== CUSTOM MODEL RESULTS =====")
    print(custom_results)

    print("\n===== SKLEARN RESULTS =====")
    print(sklearn_results)

    if task_type in ["binary", "multiclass"]:
        metrics_to_compare = ["accuracy", "precision", "recall", "f1"]
    else:
        metrics_to_compare = ["mse", "rmse", "r2"]

    rows = []
    for metric in metrics_to_compare:
        rows.append({
            "Metric": metric,
            "Custom": custom_results.get(metric, None),
            "Sklearn": sklearn_results.get(metric, None)
        })

    comparison_df = pd.DataFrame(rows)
    display(comparison_df)

Pedagogical interpretation block

In [ ]:
def pedagogical_interpretation(model, results, task_type):
    print("\n===== PEDAGOGICAL INTERPRETATION =====")

    if task_type in ["binary", "multiclass"]:
        train_metric = model.history["train_metric"][-1] if len(model.history["train_metric"]) > 0 else None
        val_metric = model.history["val_metric"][-1] if len(model.history["val_metric"]) > 0 else None

        if train_metric is not None and val_metric is not None:
            gap = train_metric - val_metric

            if train_metric < 0.7 and val_metric < 0.7:
                print("- The model may be underfitting: both train and validation accuracy are low.")
            elif gap > 0.10:
                print("- The model may be overfitting: training accuracy is much higher than validation accuracy.")
            else:
                print("- The model appears to generalize reasonably well.")

        print(f"- Final accuracy: {results.get('accuracy', None):.4f}")
        print(f"- Final F1-score: {results.get('f1', None):.4f}")

    elif task_type == "regression":
        train_metric = model.history["train_metric"][-1] if len(model.history["train_metric"]) > 0 else None
        val_metric = model.history["val_metric"][-1] if len(model.history["val_metric"]) > 0 else None

        if train_metric is not None and val_metric is not None:
            if val_metric > train_metric * 1.5:
                print("- Validation error is much larger than training error, suggesting overfitting.")
            else:
                print("- Training and validation errors are relatively close.")

        print(f"- Final RMSE: {results.get('rmse', None):.4f}")
        print(f"- Final R² score: {results.get('r2', None):.4f}")

Main training runner

In [ ]:
def run_training_pipeline(df: pd.DataFrame, config: dict):
    target_col = config["target_col"]
    task_type = config["task_type"]
    model_type = config["model_type"]

    df_model = df.copy()
    df_model = basic_clean_dataframe(df_model, drop_na=True)

    # Keep numeric features for now
    df_model = keep_numeric_features_only(df_model, target_col=target_col)

    X_train, X_test, y_train, y_test, metadata = prepare_dataset_from_dataframe(
        df=df_model,
        target_col=target_col,
        task_type=task_type,
        test_size=config["test_size"],
        standardize=config["standardize"]
    )

    model = None
    results = None

    if model_type == "historical_perceptron":
        if task_type != "binary":
            raise ValueError("Historical perceptron only supports binary classification.")
        model = HistoricalPerceptron(
            learning_rate=config["learning_rate"],
            epochs=config["epochs"]
        )
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test).ravel()
        results = Metrics.classification_metrics(y_test.ravel().astype(int), y_pred, average="binary")

    elif model_type == "activation_perceptron":
        if task_type not in ["binary", "regression"]:
            raise ValueError("Activation perceptron supports only binary classification or regression.")
        model = ActivationPerceptron(
            task_type=task_type,
            activation="sigmoid" if task_type == "binary" else "linear",
            learning_rate=config["learning_rate"],
            epochs=config["epochs"],
            l2_lambda=config["l2_lambda"]
        )
        model.fit(X_train, y_train)
        if task_type == "binary":
            y_pred = model.predict(X_test).ravel()
            results = Metrics.classification_metrics(y_test.ravel().astype(int), y_pred, average="binary")
        else:
            y_pred = model.predict(X_test).ravel()
            results = Metrics.regression_metrics(y_test.ravel(), y_pred)

    elif model_type == "mlp":
        output_dim = 1 if task_type in ["binary", "regression"] else y_train.shape[1]
        model = MLP(
            input_dim=X_train.shape[1],
            hidden_layers=config["hidden_layers"],
            output_dim=output_dim,
            task_type=task_type,
            hidden_activation=config["hidden_activation"],
            learning_rate=config["learning_rate"],
            epochs=config["epochs"],
            batch_size=config["batch_size"],
            l2_lambda=config["l2_lambda"],
            early_stopping=config["early_stopping"],
            patience=config["patience"],
            verbose=config["verbose"]
        )
        model.fit(X_train, y_train, X_val=X_test, y_val=y_test)
        results = evaluate_model(model, X_test, y_test, task_type=task_type)

    else:
        raise ValueError(f"Unsupported model type: {model_type}")

    return model, results, X_train, X_test, y_train, y_test, metadata, df_model

Automatic result display

In [ ]:
def display_training_results(model, results, X_train, X_test, y_train, y_test, config):
    task_type = config["task_type"]
    model_type = config["model_type"]

    print("===== FINAL RESULTS =====")
    print(results)

    if hasattr(model, "history"):
        if model_type == "historical_perceptron":
            if "errors" in model.history:
                plt.figure(figsize=(7, 4))
                plt.plot(model.history["errors"])
                plt.title("Historical Perceptron Errors per Epoch")
                plt.xlabel("Epoch")
                plt.ylabel("Errors")
                plt.grid(True)
                plt.show()
        else:
            Visualizer.plot_training_history(model.history, task_type=task_type)

    if task_type in ["binary", "multiclass"] and "confusion_matrix" in results:
        Visualizer.plot_confusion_matrix(results["confusion_matrix"])

    if task_type == "regression":
        if hasattr(model, "predict"):
            y_pred = model.predict(X_test).ravel()
            Visualizer.plot_regression_predictions(y_test.ravel(), y_pred)

    if X_train.shape[1] == 2 and task_type in ["binary", "multiclass"]:
        try:
            Visualizer.plot_decision_boundary(model, X_train, y_train, task_type=task_type)
        except Exception as e:
            print("Decision boundary plot skipped:", str(e))

    if model_type != "historical_perceptron":
        pedagogical_interpretation(model, results, task_type)

Full train button hookup

In [ ]:
def attach_training_handler(df: pd.DataFrame, widgets_dict):
    output_area = widgets.Output()

    def on_run_clicked(b):
        with output_area:
            output_area.clear_output()

            try:
                config = extract_config_from_widgets(widgets_dict)
                print("Running with config:")
                print(config)

                model, results, X_train, X_test, y_train, y_test, metadata, df_model = run_training_pipeline(df, config)

                display_training_results(model, results, X_train, X_test, y_train, y_test, config)

                if config["sklearn_compare"]:
                    print("\n===== BONUS: SKLEARN COMPARISON =====")
                    sklearn_results = run_sklearn_comparison(X_train, X_test, y_train, y_test, config)
                    display_comparison_results(results, sklearn_results, config["task_type"])

            except Exception as e:
                print("ERROR:", str(e))

    widgets_dict["run_button"].on_click(on_run_clicked)
    display(output_area)

Architecture comparison mode

In [ ]:
def run_architecture_comparison(df: pd.DataFrame, base_config: dict, architectures: dict):
    comparison_rows = []

    for name, hidden_layers in architectures.items():
        config = base_config.copy()
        config["model_type"] = "mlp"
        config["hidden_layers"] = hidden_layers
        config["verbose"] = False

        model, results, X_train, X_test, y_train, y_test, metadata, df_model = run_training_pipeline(df, config)

        row = {
            "Architecture": name,
            "Hidden Layers": str(hidden_layers)
        }

        if config["task_type"] in ["binary", "multiclass"]:
            row["Accuracy"] = results.get("accuracy", None)
            row["Precision"] = results.get("precision", None)
            row["Recall"] = results.get("recall", None)
            row["F1"] = results.get("f1", None)
        else:
            row["MSE"] = results.get("mse", None)
            row["RMSE"] = results.get("rmse", None)
            row["R2"] = results.get("r2", None)

        comparison_rows.append(row)

    comparison_df = pd.DataFrame(comparison_rows)
    return comparison_df

Comparison button hookup

In [ ]:
def attach_comparison_handler(df: pd.DataFrame, widgets_dict):
    output_area = widgets.Output()

    def on_compare_clicked(b):
        with output_area:
            output_area.clear_output()

            try:
                config = extract_config_from_widgets(widgets_dict)

                architectures = {
                    "Shallow": [8],
                    "Medium": [16, 16],
                    "Deep": [32, 16, 8]
                }

                comparison_df = run_architecture_comparison(df, config, architectures)

                print("===== ARCHITECTURE COMPARISON =====")
                display(comparison_df)

            except Exception as e:
                print("ERROR:", str(e))

    widgets_dict["compare_button"].on_click(on_compare_clicked)
    display(output_area)

3D visualization bonus

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

3D loss surface demo

In [ ]:
def plot_3d_loss_surface():
    w1_vals = np.linspace(-3, 3, 60)
    w2_vals = np.linspace(-3, 3, 60)
    W1, W2 = np.meshgrid(w1_vals, w2_vals)

    Z = W1**2 + W2**2 + 0.5*np.sin(3*W1)*np.cos(3*W2)

    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection='3d')
    ax.plot_surface(W1, W2, Z, cmap='viridis', alpha=0.9)
    ax.set_title("3D Loss Surface Illustration")
    ax.set_xlabel("Weight 1")
    ax.set_ylabel("Weight 2")
    ax.set_zlabel("Loss")
    plt.show()

In [ ]:
# Cell 25 — Dataset inspection tools
def inspect_dataframe(df: pd.DataFrame, target_col: Optional[str] = None):
    print("===== DATASET OVERVIEW =====")
    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")
    print("\nColumn types:")
    print(df.dtypes)

    print("\nMissing values:")
    print(df.isnull().sum())

    print("\nSummary statistics:")
    display(df.describe(include="all"))

    if target_col is not None and target_col in df.columns:
        print(f"\n===== TARGET ANALYSIS: {target_col} =====")
        print(df[target_col].value_counts(dropna=False) if df[target_col].dtype == "object" or df[target_col].nunique() < 20 else df[target_col].describe())

In [ ]:
!pip install ipywidgets

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

In [ ]:
def suggest_task_type(df, target_col):
    y = df[target_col]

    # If numeric
    if pd.api.types.is_numeric_dtype(y):
        unique_vals = y.nunique()

        # Small number of unique values → classification
        if unique_vals <= 10:
            if unique_vals == 2:
                return "binary"
            else:
                return "multiclass"
        else:
            return "regression"

    # If not numeric → classification
    else:
        unique_vals = y.nunique()
        if unique_vals == 2:
            return "binary"
        else:
            return "multiclass"

In [ ]:
w_demo = create_training_widgets(df_demo)
display_training_widgets(w_demo)

In [ ]:
# Demo dataframe for immediate UI testing
X_demo, y_demo = get_binary_dataset(dataset_type="moons", n_samples=500, noise=0.25)
df_demo = pd.DataFrame(X_demo, columns=["x1", "x2"])
df_demo["target"] = y_demo

inspect_dataframe(df_demo, target_col="target")

w_demo = create_training_widgets(df_demo)
display_training_widgets(w_demo)
attach_training_handler(df_demo, w_demo)
attach_comparison_handler(df_demo, w_demo)

In [ ]:
def basic_clean_dataframe(df: pd.DataFrame, drop_na: bool = False):
    df = df.copy()

    # Drop duplicates
    df = df.drop_duplicates()

    if drop_na:
        df = df.dropna()
    else:
        # Fill missing values
        for col in df.columns:
            if df[col].isnull().sum() > 0:
                if pd.api.types.is_numeric_dtype(df[col]):
                    df[col] = df[col].fillna(df[col].mean())
                else:
                    df[col] = df[col].fillna(df[col].mode()[0])

    return df

In [ ]:
def keep_numeric_features_only(df: pd.DataFrame, target_col: str):
    df = df.copy()

    y = df[[target_col]]
    X = df.drop(columns=[target_col])

    X_numeric = X.select_dtypes(include=["number"])

    dropped_cols = set(X.columns) - set(X_numeric.columns)
    if len(dropped_cols) > 0:
        print(f"⚠️ Dropped non-numeric columns: {dropped_cols}")

    result = pd.concat([X_numeric, y], axis=1)
    return result

In [ ]:
experiment_log = []

def log_experiment(config, metrics):
    entry = {**config, **metrics}
    experiment_log.append(entry)

def show_experiments():
    df = pd.DataFrame(experiment_log)
    display(df)

In [ ]:
def attach_training_handler(df: pd.DataFrame, widgets_dict):
    output_area = widgets.Output()

    def on_train_clicked(b):
        with output_area:
            output_area.clear_output()

            try:
                config = extract_config_from_widgets(widgets_dict)

                print("Running with config:")
                print(config)

                model, results, X_train, X_test, y_train, y_test, metadata, df_model = run_training_pipeline(df, config)

                display_training_results(model, results, X_train, X_test, y_train, y_test, config)

                # log experiment
                loggable_metrics = {}
                for k, v in results.items():
                    if k != "confusion_matrix":
                        loggable_metrics[k] = v
                log_experiment(config, loggable_metrics)

                if config["sklearn_compare"]:
                    print("\n===== BONUS: SKLEARN COMPARISON =====")
                    sklearn_results = run_sklearn_comparison(X_train, X_test, y_train, y_test, config)
                    display_comparison_results(results, sklearn_results, config["task_type"])

            except Exception as e:
                print("ERROR:", str(e))

    widgets_dict["run_button"].on_click(on_train_clicked)
    display(output_area)

Experiment 1

In [ ]:
X_demo, y_demo = get_binary_dataset(dataset_type="moons", n_samples=500, noise=0.25)
df_demo = pd.DataFrame(X_demo, columns=["x1", "x2"])
df_demo["target"] = y_demo

w_demo = create_training_widgets(df_demo)
display_training_widgets(w_demo)
attach_training_handler(df_demo, w_demo)
attach_comparison_handler(df_demo, w_demo)

In [ ]:
show_experiments()

Experiment 2

In [ ]:
X_demo, y_demo = get_binary_dataset(dataset_type="moons", n_samples=500, noise=0.25)
df_demo = pd.DataFrame(X_demo, columns=["x1", "x2"])
df_demo["target"] = y_demo

w_demo = create_training_widgets(df_demo)
display_training_widgets(w_demo)
attach_training_handler(df_demo, w_demo)
attach_comparison_handler(df_demo, w_demo)

In [ ]:
show_experiments()

Experiment 3

In [ ]:
X_demo, y_demo = get_binary_dataset(dataset_type="moons", n_samples=500, noise=0.25)
df_demo = pd.DataFrame(X_demo, columns=["x1", "x2"])
df_demo["target"] = y_demo

w_demo = create_training_widgets(df_demo)
display_training_widgets(w_demo)
attach_training_handler(df_demo, w_demo)
attach_comparison_handler(df_demo, w_demo)

In [ ]:
show_experiments()